In [ ]:
import pandas as pd
import json
import re
from pathlib import Path
import numpy as np

def clean_electrical_capacity(capacity_str):
    """
    Clean electrical capacity string and convert to integer kW
    Examples: "1,600 kW" -> 1600, "2,000 ekW" -> 2000
    """
    if not capacity_str or capacity_str.lower() in ['null', 'none', '']:
        return 0
    
    # Remove commas and extract numbers
    cleaned = re.sub(r'[,\s]', '', str(capacity_str))
    # Extract the numeric part
    match = re.search(r'(\d+)', cleaned)
    if match:
        return int(match.group(1))
    return 0

def classify_fuel_type(description):
    """
    Classify fuel type based on description
    Returns 'natural_gas', 'diesel', or 'other'
    """
    description_lower = description.lower()
    
    # Handle dual fuel specifically - check the description content
    if 'dual fuel' in description_lower:
        # For dual fuel, look for more specific indicators
        if 'natural gas' in description_lower or 'ng)' in description_lower:
            return 'natural_gas'  # Assume primary fuel is natural gas for dual fuel
        elif 'diesel' in description_lower or 'fuel oil' in description_lower:
            return 'diesel'
        else:
            return 'natural_gas'  # Default dual fuel to natural gas
    
    # Check for diesel first (more specific) - EXPANDED LIST
    diesel_keywords = ['diesel', 'gen-set', 'distillate oil', 'fuel oil', 'emergency generator', 
                       'caterpillar', 'cummins', 'backup']
    for keyword in diesel_keywords:
        if keyword in description_lower:
            return 'diesel'
    
    # Check for natural gas indicators (be more specific)
    natural_gas_keywords = ['natural gas', 'natural-gas', 'gas-fired', 'gas fired', 'ng)', 'natural gas-fired']
    for keyword in natural_gas_keywords:
        if keyword in description_lower:
            return 'natural_gas'
    
    # Check for just "gas" but exclude common false positives
    if 'gas' in description_lower and 'diesel' not in description_lower and 'exhaust' not in description_lower:
        return 'natural_gas'
    
    return 'other'

def analyze_datacenter_json(json_file_path):
    """
    Analyze a single JSON file and return datacenter info with fuel breakdown
    """
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Extract basic info
        datacenter_info = {
            'datacenter_name': data.get('dataCenterName', ''),
            'permit_issuance_date': data.get('permitIssuanceDate', ''),
            'registration_number': data.get('registrationNumber', ''),
            'location': data.get('location', ''),
            'source_file': json_file_path.name
        }
        
        # Initialize fuel totals
        diesel_capacity = 0
        natural_gas_capacity = 0
        other_capacity = 0
        
        # Analyze equipment
        equipment_summary = data.get('equipmentSummary', [])
        
        for equipment in equipment_summary:
            # Get electrical capacity
            electrical_capacity_str = equipment.get('electricalCapacity_kW', '')
            capacity_kw = clean_electrical_capacity(electrical_capacity_str)
            
            # Skip if no capacity or capacity is 0
            if capacity_kw == 0:
                continue
            
            # Get description and check for Burnham
            description = equipment.get('description', '')
            
            # Check if "burnham" is in description (case-insensitive)
            if 'burnham' in description.lower():
                # Split capacity 50/50 between gas and diesel
                half_capacity = capacity_kw / 2
                diesel_capacity += half_capacity
                natural_gas_capacity += half_capacity
                print(f"BURNHAM SPLIT: {description} -> {capacity_kw} kW split to {half_capacity} diesel + {half_capacity} gas")
            else:
                # Normal classification
                fuel_type = classify_fuel_type(description)
                # Add to appropriate total
                if fuel_type == 'diesel':
                    diesel_capacity += capacity_kw
                    print(f"  -> ADDED TO DIESEL: {capacity_kw} kW (total diesel now: {diesel_capacity})")
                elif fuel_type == 'natural_gas':
                    natural_gas_capacity += capacity_kw
                    print(f"  -> ADDED TO NATURAL GAS: {capacity_kw} kW (total gas now: {natural_gas_capacity})")
                else:
                    other_capacity += capacity_kw
                    print(f"  -> ADDED TO OTHER: {capacity_kw} kW (total other now: {other_capacity})")
                # Add to appropriate total
                if fuel_type == 'diesel':
                    diesel_capacity += capacity_kw
                elif fuel_type == 'natural_gas':
                    natural_gas_capacity += capacity_kw
                else:
                    other_capacity += capacity_kw
                    
                # Debug print for MMBTU equipment
                if 'mmbtu' in str(equipment).lower():
                    print(f"MMBTU EQUIPMENT: {description} -> {fuel_type} -> {capacity_kw} kW")
        
        # Calculate total and percentages
        total_capacity = diesel_capacity + natural_gas_capacity + other_capacity
        
        datacenter_info.update({
            'diesel_capacity_kw': diesel_capacity,
            'natural_gas_capacity_kw': natural_gas_capacity,
            'other_capacity_kw': other_capacity,
            'total_capacity_kw': total_capacity,
            'diesel_percentage': (diesel_capacity / total_capacity * 100) if total_capacity > 0 else 0,
            'natural_gas_percentage': (natural_gas_capacity / total_capacity * 100) if total_capacity > 0 else 0,
            'other_percentage': (other_capacity / total_capacity * 100) if total_capacity > 0 else 0
        })
        
        return datacenter_info
        
    except Exception as e:
        print(f"Error processing {json_file_path}: {e}")
        return None
def generate_fuel_analysis_csv():
    """
    Generate CSV files with datacenter fuel analysis
    """
    # Find all JSON files in json_data directory
    json_data_path = Path('json_data')
    
    if not json_data_path.exists():
        print("Error: json_data directory not found!")
        return
    
    # Collect all JSON files from all year subdirectories
    json_files = []
    for year_dir in json_data_path.iterdir():
        if year_dir.is_dir():
            json_files.extend(year_dir.glob("*.json"))
    
    # Filter out the combined files (all_extracted_data_*.json) if any exist
    json_files = [f for f in json_files if not f.name.startswith('all_extracted_data_')]
    
    print(f"Found {len(json_files)} JSON files to analyze")
    
    # Analyze each file
    results = []
    for json_file in json_files:
        print(f"Processing: {json_file}")
        result = analyze_datacenter_json(json_file)
        if result:
            results.append(result)
    
    if not results:
        print("No valid data found!")
        return
    
    # Create DataFrame
    df = pd.DataFrame(results)
    
    # Sort by datacenter name
    df = df.sort_values('datacenter_name')
    
    # Reorder columns for better readability
    column_order = [
        'datacenter_name', 'permit_issuance_date', 'registration_number', 'location',
        'total_capacity_kw', 'diesel_capacity_kw', 'natural_gas_capacity_kw', 'other_capacity_kw',
        'diesel_percentage', 'natural_gas_percentage', 'other_percentage', 'source_file'
    ]
    
    df = df[column_order]
    
    # Round percentages to 2 decimal places
    percentage_columns = ['diesel_percentage', 'natural_gas_percentage', 'other_percentage']
    df[percentage_columns] = df[percentage_columns].round(2)
    
    # Save main data to CSV
    output_file = 'datacenter_fuel_analysis.csv'
    df.to_csv(output_file, index=False)
    
    # Create and save summary data to separate CSV
    summary_data = {
        'Metric': [
            'Total Datacenters',
            'Total Electrical Capacity (kW)',
            'Total Diesel Capacity (kW)',
            'Total Natural Gas Capacity (kW)',
            'Total Other Capacity (kW)',
            'Overall Diesel Percentage',
            'Overall Natural Gas Percentage',
            'Overall Other Percentage'
        ],
        'Value': [
            len(df),
            df['total_capacity_kw'].sum(),
            df['diesel_capacity_kw'].sum(),
            df['natural_gas_capacity_kw'].sum(),
            df['other_capacity_kw'].sum(),
            round((df['diesel_capacity_kw'].sum() / df['total_capacity_kw'].sum() * 100), 2) if df['total_capacity_kw'].sum() > 0 else 0,
            round((df['natural_gas_capacity_kw'].sum() / df['total_capacity_kw'].sum() * 100), 2) if df['total_capacity_kw'].sum() > 0 else 0,
            round((df['other_capacity_kw'].sum() / df['total_capacity_kw'].sum() * 100), 2) if df['total_capacity_kw'].sum() > 0 else 0
        ]
    }
    
    summary_df = pd.DataFrame(summary_data)
    summary_output_file = 'datacenter_fuel_analysis_summary.csv'
    summary_df.to_csv(summary_output_file, index=False)
    
    print(f"\n✓ Analysis complete! Results saved to:")
    print(f"  - Main data: {output_file}")
    print(f"  - Summary: {summary_output_file}")
    print(f"Processed {len(results)} datacenters")
    print(f"Total electrical capacity: {df['total_capacity_kw'].sum():,} kW")
    print(f"Diesel: {df['diesel_capacity_kw'].sum():,} kW ({df['diesel_capacity_kw'].sum() / df['total_capacity_kw'].sum() * 100:.1f}%)")
    print(f"Natural Gas: {df['natural_gas_capacity_kw'].sum():,} kW ({df['natural_gas_capacity_kw'].sum() / df['total_capacity_kw'].sum() * 100:.1f}%)")
    print(f"Other: {df['other_capacity_kw'].sum():,} kW ({df['other_capacity_kw'].sum() / df['total_capacity_kw'].sum() * 100:.1f}%)")

if __name__ == "__main__":
    generate_fuel_analysis_csv()

In [2]:
import json
from pathlib import Path
from collections import Counter

def get_unique_operational_categories():
    """
    Get a simple list of all unique operational limit categories
    """
    json_data_path = Path('json_data')
    categories = set()
    
    # Collect all individual JSON files
    json_files = []
    for year_dir in json_data_path.iterdir():
        if year_dir.is_dir():
            json_files.extend([f for f in year_dir.glob("*.json") 
                             if not f.name.startswith('all_extracted_data_')])
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            operational_limits = data.get('operationalLimits', [])
            for limit in operational_limits:
                category = limit.get('category', '')
                if category:
                    categories.add(category)
        
        except Exception as e:
            continue
    
    # Print sorted list
    unique_categories = sorted(categories)
    print("Unique Operational Limit Categories:")
    print("=" * 40)
    for i, category in enumerate(unique_categories, 1):
        print(f"{i:2d}. {category}")
    
    print(f"\nTotal: {len(unique_categories)} unique categories")

get_unique_operational_categories()

Unique Operational Limit Categories:
 1. Annual Emission Limit
 2. Annual Operational Constraint
 3. Blackout Testing Limit
 4. Conditional SCR Operation/Installation
 5. Construction Timing/BACT Review
 6. Control Device Operating Limit
 7. Control Device Operating Requirement
 8. Control Device Operating Temperature
 9. Control Device Operation
10. Control Device Operation (SCR Temperature)
11. Control Device Specification
12. Control Equipment Operation
13. Cooling Tower Limit
14. Electrical Capacity Limit
15. Electrical Output
16. Electrical Output Limit
17. Electrical Power Output
18. Emergency Power Generation
19. Emission Cap
20. Emissions Cap
21. Engine Electrical Power Output Limit
22. Engine Load
23. Fuel Certification
24. Fuel Certification/Monitoring
25. Fuel Monitoring Recalibration
26. Fuel Specification
27. Fuel Specification (ASTM)
28. Fuel Specification (Cetane/Aromatic)
29. Fuel Specification (Sulfur Content)
30. Fuel Specification (Sulfur)
31. Fuel Throughput
32. Fue